In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API",
    category=UserWarning,
)

In [ ]:
import pymc as pm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import arviz as az
from helpers import ( prep_the_data, 
                      export_puma_kepler, 
                          make_daily_table_for_model_with_nta,
                      load_idata,
                        make_typical_week_2025,
                        make_daily_observed_2025
                    )


import geopandas as gpd
from keplergl import KeplerGl

In [ ]:
# Rebuild daily_df_train exactly as in training
daily_df_train, coords = make_daily_table_for_model_with_nta(
    df_train_2022_2024,
    complaint_value=COMPLAINT,
)



In [ ]:
# Rebuild puma → nta mapping
puma_nta_map = (
    daily_df_train[["puma_idx", "nta_idx"]]
    .drop_duplicates()
    .sort_values("puma_idx")
)


In [ ]:

puma_to_nta_idx = puma_nta_map["nta_idx"].to_numpy()


In [ ]:
lam_post = idata_nta.posterior["lam"]

# Posterior mean forecast
df_forecast = (
    lam_post
    .mean(dim=("chain", "draw"))
    .to_dataframe(name="lam_forecast")
    .reset_index()
)


In [ ]:
hdi = az.hdi(lam_post, hdi_prob=0.9)["lam"]

hdi_df = (
    hdi.to_dataframe(name="lam_hdi")
       .reset_index()
       .pivot_table(
           index=["puma", "dow"],
           columns="hdi",
           values="lam_hdi",
       )
       .reset_index()
       .rename(columns={"lower": "lam_low_90", "higher": "lam_high_90"})
)

df_forecast = df_forecast.merge(hdi_df, on=["puma", "dow"], how="left")
df_forecast["lam_width_90"] = (
    df_forecast["lam_high_90"] - df_forecast["lam_low_90"]
)


In [ ]:
typical_2025 = make_typical_week_2025(
    df_311_2025,
    complaint_col="descriptor_group"
)


In [ ]:
cmp_2025 = (
    df_forecast
    .merge(typical_2025, on=["puma", "dow"], how="left")
)

cmp_2025["error"] = cmp_2025["observed_2025"] - cmp_2025["lam_forecast"]
cmp_2025["abs_error"] = cmp_2025["error"].abs()

cmp_2025["within_90"] = (
    (cmp_2025["observed_2025"] >= cmp_2025["lam_low_90"]) &
    (cmp_2025["observed_2025"] <= cmp_2025["lam_high_90"])
)


In [ ]:
print(f"MAE: {cmp_2025["abs_error"].mean()}")
print(f"Median AE: {cmp_2025["abs_error"].median()}")
print(f"90% Coverage: {mp_2025["within_90"].mean()}")

In [ ]:
# Forecast is keyed by (puma, dow), observed is keyed by (puma, date, dow)
daily_cmp_2025 = daily_2025.merge(
    df_forecast,
    on=["puma", "dow"],
    how="left",
)

# Useful comparison columns
daily_cmp_2025["error"] = daily_cmp_2025["daily_count"] - daily_cmp_2025["lam_forecast"]
daily_cmp_2025["abs_error"] = daily_cmp_2025["error"].abs()

# If you have intervals:
if "lam_low_90" in daily_cmp_2025.columns and "lam_high_90" in daily_cmp_2025.columns:
    daily_cmp_2025["within_90"] = (
        (daily_cmp_2025["daily_count"] >= daily_cmp_2025["lam_low_90"]) &
        (daily_cmp_2025["daily_count"] <= daily_cmp_2025["lam_high_90"])
    )


In [ ]:
gdf_daily_kepler = kepler_typical_week_from_dow_complaint(
    daily_cmp_2025,
    puma_geojson_path="../data/raw/nyc/geographies/nyc_pumas_2020.geojson",
    out_path="data/processed/kepler/model3_daily_forecast_vs_2025.geojson",
)
